# Manuscript Figures 3 to 6 and Supplementary Figure 9, one visual style (v4)

v4 changes only two labels from v3: one condition label in Figure 4b moved clear of its neighbour, and the Supplementary Figure 9 panel captions read 'Pair ahead at every K tested' where the break-even K and its interval are all below the grid.

Redraws Figures 3 to 6 so they match Figures 1, 2 and 7, and draws the break-even figure from notebook 18b as Supplementary Figure 9. No model is run and no number changes: the data cells are copied unchanged from `FIGURES_brset`, and the last cell checks the recomputed numbers against `final_figures_data_v2.json`.

**One encoding throughout**
- Where populations are compared (Figures 4 and 6), colour is the domain (retina blue, skin vermilion, as in Figures 1 and 2) and marker shape is the population.
- Where relationships between two systems are compared (Figures 3 and 5), colour is the relationship, with the same five colours in both figures.
- Numbers are written as point (low to high), as in Figure 2.

The figures they replace are renamed with the prefix `SUPERSEDED_` in `05_figures/manuscript_final/`.

BRSET is credentialed data: it is read only inside this runtime, and only figures and aggregate numbers are written. CPU, High-RAM; about 20 to 30 minutes.

In [1]:
# Cell 1: load both domains
from google.colab import drive
drive.mount('/content/drive')
import os, json, itertools, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'; DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
OUT = f'{BASE}/05_figures/manuscript_final'; os.makedirs(OUT, exist_ok=True)
def pick(df, c): return next(x for x in c if x in df.columns)
def tag(k): return '' if k == 0 else f'_seed{k}'
PAPER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
         'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']
CK = {'densenet121': 'densenet', 'efficientnet_b3': 'effnet', 'resnet50': 'resnet', 'vit_base': 'vit', 'retfound': 'retfound'}
ck_of = lambda v: next(c for p, c in CK.items() if v.startswith(p))
def load(path, ids, col=None):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = col or pick(df, ['predicted_probability','probability','prob'])
    s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(ids); assert s.notna().all(), path; return s.to_numpy()
DOM = {}
fr = pd.read_csv(f'{BASE}/01_splits/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
dd = pd.read_csv(f'{BASE}/01_splits/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
for site, df, tpl, f in (('EyePACS', fr, 'eyepacs_predictions_fresh_{m}_v2.csv', 0.25), ('DDR', dd, 'ddr_predictions_{m}_v2.csv', None)):
    ids = df.image_id.to_numpy(); g = df[pick(df, ['level','grade'])].to_numpy().astype(int)
    M = {f'{v}{tag(k)}': {'s': load(f'{BASE}/03_predictions/' + tpl.format(m=f'{v}{tag(k)}'), ids), 'ckpt': ck_of(v), 'variant': v} for v in PAPER for k in (0, 1, 2)}
    DOM[site] = {'domain': 'retina', 'models': M, 'pos': g >= 2, 'classes': {'moderate': g == 2, 'severe NPDR': g == 3, 'PDR': g == 4},
                 'f': f if f else 1.31 * float((dd.grade[dd.half == 'A'] >= 2).mean())}
CLASSES8 = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
for es in ('HIBA', 'PAD'):
    D = pd.read_csv(f'{DERM}/manifest_{es}.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
    D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True); ids = D.image_id.to_numpy()
    M = {}
    for s in ('HAM10000', 'BCN20000'):
        for a in ('resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2'):
            for k in (0, 1): M[f'{s}_{a}_seed{k}'] = {'s': load(f'{DERM}/predictions/{es}__{s}_{a}_seed{k}.csv', ids, 'p_malignant'), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
    mal = D.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy()
    DOM[es] = {'domain': 'dermatology', 'models': M, 'pos': mal, 'classes': {'BCC': (D.dx == 'BCC').to_numpy(), 'SCC': (D.dx == 'SCC').to_numpy(), 'melanoma': (D.dx == 'MEL').to_numpy()},
               'f': 1.31 * mal.mean()}
# BRSET (credentialed data: stays in this runtime; only figures and aggregate numbers are written)
import hashlib
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
BR = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'
assert sha(f'{BR}/label_brset.csv') == json.load(open(f'{BR}/brset_access_record.json'))['label_file_sha256']
L = pd.read_csv(f'{BR}/label_brset.csv', dtype=str); gB = pd.to_numeric(L['DR_ICDR'], errors='coerce')
L = L[gB.isin([0, 1, 2, 3, 4])].copy(); L['grade'] = gB[gB.isin([0, 1, 2, 3, 4])].astype(int)
L['image_id'] = L['image_id'].astype(str).str.replace(r'\.(jpg|jpeg|png)$', '', regex=True); L = L.sort_values('image_id').reset_index(drop=True)
M = {}
for s in ('eyepacs', 'ddr'):
    for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class'):
        for k in (0, 1, 2):
            M[f'{s}_{a}{tag(k)}'] = {'s': load(f'{BASE}/03_predictions/brset_predictions_{s}_{a}{tag(k)}_v2.csv', L.image_id.to_numpy()), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
gb = L.grade.to_numpy()
DOM['BRSET'] = {'domain': 'retina', 'models': M, 'pos': gb >= 2, 'classes': {'moderate': gb == 2, 'severe NPDR': gb == 3, 'PDR': gb == 4}, 'f': 1.31 * float((gb >= 2).mean())}
for k, v in DOM.items(): print(k, len(v['models']), 'models, f =', round(v['f'], 3), {c: int(m.sum()) for c, m in v['classes'].items()})

Mounted at /content/drive
EyePACS 33 models, f = 0.25 {'moderate': 2373, 'severe NPDR': 384, 'PDR': 328}
DDR 33 models, f = 0.555 {'moderate': 4477, 'severe NPDR': 236, 'PDR': 913}
HIBA 24 models, f = 0.609 {'BCC': 340, 'SCC': 158, 'melanoma': 253}
PAD 24 models, f = 0.62 {'BCC': 839, 'SCC': 190, 'melanoma': 52}
BRSET 24 models, f = 0.074 {'moderate': 451, 'severe NPDR': 78, 'PDR': 396}


In [2]:
# Cell 2: per ordered pair and class: partner's own miss rate, union rescue share, matched-volume rescue share; pair relationship and referral agreement
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
def relation(site, A, B):
    if 'source' not in A:
        if A['ckpt'] != B['ckpt']: return 'different design'
        return 'seed sibling' if A['variant'] == B['variant'] else 'variant sibling'
    if A['variant'] == B['variant']: return 'seed sibling'
    if A['ckpt'] == B['ckpt']: return 'same checkpoint, different data'
    return 'different design, same data' if A['source'] == B['source'] else 'different design, different data'
ROWS = []
for site, S in DOM.items():
    M = S['models']; names = list(M); n = len(S['pos'])
    R = {m: refer(M[m]['s'], S['f']) for m in names}; RK = {m: rankdata(M[m]['s']) / n for m in names}
    for a, b in itertools.permutations(names, 2):
        e = refer(np.maximum(RK[a], RK[b]), S['f']); rel = relation(site, M[a], M[b]); kap = kappa(R[a], R[b])
        for c, y in S['classes'].items():
            ma, mb = y & ~R[a], y & ~R[b]; na = ma.sum()
            if na == 0: continue
            ROWS.append({'site': site, 'domain': S['domain'], 'a': a, 'b': b, 'relationship': rel, 'kappa': kap, 'class': c,
                         'partner_miss_rate': mb.sum() / y.sum(), 'anchor_miss_rate': na / y.sum(),
                         'union_rescue_share': (ma & ~mb).sum() / na, 'mv_rescue_share': (na - (y & ~e).sum()) / na,
                         'lift': ((ma & mb).sum() / y.sum()) / ((na / y.sum()) * (mb.sum() / y.sum())) if mb.sum() else np.nan})
PR = pd.DataFrame(ROWS); print(PR.groupby(['site', 'class']).size().to_string())
# rank correlation for every unordered pair (Figure 5, lower row)
PK = PR.drop_duplicates(['site', 'a', 'b'])[['site', 'domain', 'a', 'b', 'relationship', 'kappa']]; PK = PK[PK.a < PK.b].copy()
RANKS = {}
for site, S in DOM.items():
    n = len(S['pos']); RK = {m: rankdata(S['models'][m]['s']) / n for m in S['models']}
    RANKS[site] = {(a, b): float(np.corrcoef(RK[a], RK[b])[0, 1]) for a, b in itertools.combinations(sorted(S['models']), 2)}
PK['rank_corr'] = [RANKS[r.site][(r.a, r.b)] for r in PK.itertuples()]
CLASSN = {(site, c): int(m.sum()) for site, S in DOM.items() for c, m in S['classes'].items()}
print('pairs:', PK.groupby('site').size().to_dict())

site     class      
BRSET    PDR             552
         moderate        552
         severe NPDR     529
DDR      PDR            1056
         moderate       1056
         severe NPDR     960
EyePACS  PDR            1056
         moderate       1056
         severe NPDR    1056
HIBA     BCC             552
         SCC             552
         melanoma        552
PAD      BCC             552
         SCC             552
         melanoma        552
pairs: {'BRSET': 276, 'DDR': 528, 'EyePACS': 528, 'HIBA': 276, 'PAD': 276}


In [3]:
# Shared style, matching Figures 1, 2 and 7
import shutil
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#333333', 'axes.linewidth': 0.8, 'xtick.color': '#333333', 'ytick.color': '#333333',
                     'axes.labelcolor': '#222222', 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.labelsize': 8.5, 'legend.frameon': False})
BLUE, SKY, ORANGE, VERM, PINK, GREY, INK = '#0072B2', '#56B4E9', '#E69F00', '#D55E00', '#CC79A7', '#8C8C8C', '#222222'
DOMCOL = {'retina': BLUE, 'dermatology': VERM}
SITES = ['EyePACS', 'DDR', 'BRSET', 'HIBA', 'PAD']
SNAME = {'EyePACS': 'EyePACS (US)', 'DDR': 'DDR (China)', 'BRSET': 'BRSET (Brazil)', 'HIBA': 'HIBA (Argentina)', 'PAD': 'PAD-UFES-20 (Brazil)'}
SDOM = {'EyePACS': 'retina', 'DDR': 'retina', 'BRSET': 'retina', 'HIBA': 'dermatology', 'PAD': 'dermatology'}
MARK = {'EyePACS': 'o', 'DDR': 's', 'BRSET': 'P', 'HIBA': '^', 'PAD': 'D'}
LS = {'EyePACS': '-', 'DDR': '--', 'BRSET': ':', 'HIBA': '-', 'PAD': '--'}
CNAME = {'moderate': 'Moderate NPDR', 'severe NPDR': 'Severe NPDR', 'PDR': 'PDR', 'BCC': 'BCC', 'SCC': 'SCC', 'melanoma': 'Melanoma'}
# relationship of two systems: one display name and one colour, used in Figures 3, 5 and 6
RELMAP = {'different design, different data': 'Different design, different data', 'different design, same data': 'Different design, same data',
          'different design': 'Different design, same data', 'same checkpoint, different data': 'Same design, different data',
          'variant sibling': 'Variant sibling (same design and data)', 'seed sibling': 'Seed sibling (repeat run)'}
RELS = ['Different design, different data', 'Different design, same data', 'Same design, different data', 'Variant sibling (same design and data)', 'Seed sibling (repeat run)']
RELCOL = dict(zip(RELS, [BLUE, SKY, ORANGE, PINK, VERM]))
def ci_txt(m, lo, hi, d=2, pct=False):
    u = '%' if pct else ''
    return f'{m:.{d}f}{u} ({lo:.{d}f} to {hi:.{d}f})'
def panel(ax, letter, text, sub=None):
    ax.set_title(f'{letter}  {text}', loc='left', fontsize=9.5, weight='bold', pad=14 if sub else 6)
    if sub: ax.text(0, 1.015, sub, transform=ax.transAxes, fontsize=8, color='#444444', va='bottom')
def title(fig, t): fig.suptitle(t, x=0.01, ha='left', fontsize=11, weight='bold')
def save(fig, name):
    for ext in ('png', 'pdf'):
        p = f'{OUT}/{name}.{ext}'
        if os.path.exists(p) and not os.path.exists(f'{OUT}/SUPERSEDED_{name}.{ext}'): shutil.move(p, f'{OUT}/SUPERSEDED_{name}.{ext}')
    fig.savefig(f'{OUT}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight'); plt.close(fig); print('saved', name)
PK['rel'] = PK.relationship.map(RELMAP); PR['rel'] = PR.relationship.map(RELMAP)
DR = json.load(open(f'{DERM}/derm_results_v1.json'))
RESULTS = f'{BASE}/04_results'

In [4]:
# Figure 3: shared missed cancers by relationship (skin, confirmatory)
TYPES = ['seed_siblings', 'diff_arch_same_source', 'same_arch_diff_source', 'diff_arch_diff_source']
TLAB = {'seed_siblings': 'Same design, same data\n(repeat runs)', 'diff_arch_same_source': 'Different design,\nsame data',
        'same_arch_diff_source': 'Same design,\ndifferent data', 'diff_arch_diff_source': 'Different design,\ndifferent data'}
TCOL = {'seed_siblings': RELCOL['Seed sibling (repeat run)'], 'diff_arch_same_source': RELCOL['Different design, same data'],
        'same_arch_diff_source': RELCOL['Same design, different data'], 'diff_arch_diff_source': RELCOL['Different design, different data']}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
F3 = {}
for ax, key, lt, ttl in zip(axes, ('HIBA_mel', 'HIBA_mal', 'PAD_mal'), 'abc', ('HIBA (Argentina): melanoma, primary', 'HIBA (Argentina): any skin cancer', 'PAD-UFES-20 (Brazil): any skin cancer')):
    r = DR['H3_training_data'][key]; hi_all = max(r[t]['ci95'][1] for t in TYPES)
    for i, t in enumerate(TYPES):
        m, (lo, hi) = r[t]['mean'], r[t]['ci95']
        ax.plot([lo, hi], [i, i], color=TCOL[t], lw=2.2, solid_capstyle='round'); ax.plot(m, i, 'o', color=TCOL[t], ms=7.5, mec='white', mew=0.8, zorder=3)
        ax.text(hi + 0.04 * hi_all, i, ci_txt(m, lo, hi), va='center', fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels([TLAB[t] for t in TYPES] if ax is axes[0] else [''] * 4); ax.invert_yaxis(); ax.set_ylim(3.5, -0.6)
    ax.axvline(1, color='#555555', lw=0.8, ls=':'); ax.set_xlim(0.8, hi_all * 1.42); ax.set_xlabel('Shared-miss ratio (1 = independent errors)')
    c_ = r['diff_arch_same_source__minus__diff_arch_diff_source']; F3[key] = c_
    panel(ax, lt, ttl, f"Same data minus different data: {ci_txt(c_['mean'], *c_['ci95'])}")
title(fig, 'Figure 3. Systems trained on different data share fewer missed cancers (95% intervals resample patients)')
fig.tight_layout(); save(fig, 'Fig3_shared_misses_by_relationship')

saved Fig3_shared_misses_by_relationship


In [5]:
# Figure 4: the miss-rate law (condition-population points with fewer than 10 misses per model omitted, as in FIGURES_brset)
PR['anchor_misses'] = PR.anchor_miss_rate * PR.apply(lambda r: CLASSN[(r.site, r['class'])], axis=1)
ND0 = PR[PR.relationship.str.startswith('different design')]
MED = ND0.groupby(['site', 'class']).anchor_misses.median()
KEEP = MED[MED >= 10].index; DROPPED = [f'{s} {c} ({m:.1f})' for (s, c), m in MED[MED < 10].items()]
ND = ND0[ND0.set_index(['site', 'class']).index.isin(KEEP)]
print('omitted (median misses per model < 10):', DROPPED)
SUMM = ND.groupby(['site', 'domain', 'class']).agg(partner_miss_rate=('partner_miss_rate', 'median'), union_rescue=('union_rescue_share', 'median'),
                                                  mv_rescue=('mv_rescue_share', 'median'), lift=('lift', 'median'), n_pairs=('a', 'size')).reset_index()
rho = spearmanr(SUMM.partner_miss_rate, SUMM.mv_rescue)[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={'width_ratios': [1, 1.05]})
ax = axes[0]; smp = ND.sample(min(len(ND), 6000), random_state=1)
for dom, col in DOMCOL.items():
    s = smp[smp.domain == dom]; ax.scatter(s.partner_miss_rate, s.union_rescue_share, s=4, alpha=0.16, color=col, lw=0, rasterized=True)
p = np.linspace(0.005, 0.6, 200)
for L, ls in ((2, ':'), (5, '--'), (10, '-.')):
    y = np.clip(1 - L * p, 0, 1); ax.plot(p, y, ls, color='#444444', lw=1)
    k = np.argmin(np.abs(y - 0.08)); ax.text(p[k] + 0.008, 0.1, f'lift {L}', fontsize=7.5, color='#444444')
ax.set_xlim(0, 0.62); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Partner's own miss rate for that condition"); ax.set_ylabel("Share of the deployed system's misses the partner catches")
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=c, ms=5, label=l) for l, c in (('Retina pairs', BLUE), ('Skin pairs', VERM))], fontsize=7.5, loc='upper right')
panel(ax, 'a', 'Every pair and condition (different designs)', 'Curves: what independence would give at each lift (shared-miss ratio)')
LABOFF = {('DDR', 'moderate'): (-7, -9, 'right')}   # moves one label clear of its neighbour
ax = axes[1]
for _, r in SUMM.iterrows():
    x, y = r.partner_miss_rate, 100 * r.mv_rescue
    ax.scatter(x, y, s=90 if r.site == 'BRSET' else 70, marker=MARK[r.site], color=DOMCOL[r.domain], edgecolor='white', lw=0.4 if r.site == 'BRSET' else 0.8, zorder=3)
    dx, dy, ha = LABOFF.get((r.site, r['class']), (6, 2, 'left'))
    ax.annotate(CNAME[r['class']], (x, y), textcoords='offset points', xytext=(dx, dy), ha=ha, fontsize=7.2, color='#333333')
ax.set_xlabel('Median miss rate of each system for that condition'); ax.set_ylabel('Median % of misses rescued at matched volume')
ax.set_xlim(0, max(SUMM.partner_miss_rate) * 1.18); ax.set_ylim(0, max(100 * SUMM.mv_rescue) * 1.12)
h = [Line2D([], [], marker=MARK[s], ls='', color=DOMCOL[SDOM[s]], mec='white', ms=7, label=SNAME[s]) for s in SITES]
ax.legend(handles=h, fontsize=7.5, loc='upper right')
panel(ax, 'b', 'Rescue follows how rarely each system misses, not severity', f'One point per condition and population; Spearman rho {rho:.2f}')
title(fig, f'Figure 4. A second reader rescues most where each reader alone rarely misses (retina and skin; {len(SUMM)} condition-population points)')
fig.tight_layout(); save(fig, 'Fig4_miss_rate_law')
print(SUMM.round(3).to_string())

omitted (median misses per model < 10): ['BRSET severe NPDR (5.0)', 'DDR severe NPDR (2.0)']
saved Fig4_miss_rate_law
       site       domain        class  partner_miss_rate  union_rescue  mv_rescue   lift  n_pairs
0     BRSET       retina          PDR              0.266         0.451      0.097  1.788      432
1     BRSET       retina     moderate              0.427         0.286      0.057  1.398      432
2       DDR       retina          PDR              0.053         0.560      0.250  8.669      864
3       DDR       retina     moderate              0.173         0.435      0.118  3.324      864
4   EyePACS       retina          PDR              0.055         0.650      0.400  7.130      864
5   EyePACS       retina     moderate              0.351         0.316      0.076  1.963      864
6   EyePACS       retina  severe NPDR              0.068         0.456      0.167  8.234      864
7      HIBA  dermatology          BCC              0.062         0.810      0.308  3.333      480


In [6]:
# Figure 5: related systems give themselves away (agreement by relationship, five populations; relative rule as in FIGURES_brset)
from sklearn.metrics import roc_auc_score
DET = {}
fig, axes = plt.subplots(2, 5, figsize=(17, 7.4), sharey=True)
rng = np.random.default_rng(0)
for row, (metric, lab) in enumerate((('kappa', 'Agreement on referral decisions (kappa)'), ('rank_corr', 'Agreement on score rankings (rank correlation)'))):
    for ax, site in zip(axes[row], SITES):
        s = PK[PK.site == site]; sib = s.relationship.str.contains('sibling'); dif = s.relationship.str.startswith('different design')
        thr = float(np.percentile(s[metric][dif], 95)); flag = float((s[metric][sib] > thr).mean())
        auc = float(roc_auc_score(np.r_[np.ones(sib.sum()), np.zeros(dif.sum())], np.r_[s[metric][sib], s[metric][dif]]))
        DET.setdefault(site, {})[metric] = {'auc_siblings_vs_different_design': round(auc, 3), 'relative_threshold': round(thr, 4), 'siblings_flagged': round(flag, 3)}
        for i, rl in enumerate(RELS):
            v = s[metric][s.rel == rl].to_numpy()
            if len(v) == 0:
                ax.text(0.02, i, 'not in this design', transform=ax.get_yaxis_transform(), va='center', fontsize=7, color='#9a9a9a'); continue
            bp = ax.boxplot([v], positions=[i], vert=False, widths=0.55, patch_artist=True, showfliers=False, whis=(2.5, 97.5),
                            medianprops=dict(color=INK, lw=1.4), whiskerprops=dict(color='#555555', lw=0.8), capprops=dict(color='#555555', lw=0.8),
                            boxprops=dict(facecolor=RELCOL[rl], alpha=0.35, edgecolor=RELCOL[rl], lw=1))
            jit = rng.uniform(-0.18, 0.18, len(v)) if len(v) > 1 else np.zeros(1)
            ax.scatter(v, i + jit, s=4 if len(v) > 200 else 7, color=RELCOL[rl], alpha=0.45 if len(v) > 200 else 0.8, lw=0, zorder=3, rasterized=True)
        ax.axvline(thr, color=INK, ls='--', lw=1)
        ax.set_yticks(range(len(RELS))); ax.set_yticklabels(RELS); ax.set_ylim(len(RELS) - 0.5, -0.6)
        ax.set_xlabel(lab, fontsize=8); ax.tick_params(axis='y', length=0)
        ax.set_title(f'{SNAME[site]}', loc='left', fontsize=9, weight='bold', pad=14)
        ax.text(0, 1.015, f'AUC {auc:.2f}; {100 * flag:.0f}% of siblings flagged', transform=ax.transAxes, fontsize=7.5, color='#444444', va='bottom')
for row, (lt, t) in enumerate((('a', 'Agreement on referral decisions'), ('b', 'Agreement on score rankings'))):
    axes[row][0].annotate(f'{lt}  {t}', xy=(0, 1), xycoords='axes fraction', xytext=(-205, 30), textcoords='offset points', fontsize=10, weight='bold', ha='left')
fig.legend(handles=[Line2D([], [], color=INK, ls='--', lw=1, label='Flag line: 95th percentile of agreement among systems known to differ in design')],
           loc='lower left', bbox_to_anchor=(0.01, -0.01), fontsize=8)
title(fig, 'Figure 5. Related systems give themselves away on local unlabelled images')
fig.tight_layout(rect=(0, 0.03, 1, 0.97)); fig.subplots_adjust(hspace=0.55); save(fig, 'Fig5_siblings'); print(json.dumps(DET, indent=1))

saved Fig5_siblings
{
 "EyePACS": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.5979,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6886,
   "siblings_flagged": 1.0
  }
 },
 "DDR": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6478,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.8423,
   "siblings_flagged": 1.0
  }
 },
 "BRSET": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.879,
   "relative_threshold": 0.6721,
   "siblings_flagged": 0.625
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 0.998,
   "relative_threshold": 0.659,
   "siblings_flagged": 1.0
  }
 },
 "HIBA": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.947,
   "relative_threshold": 0.6856,
   "siblings_flagged": 0.833
  },
  "rank_corr": {
   "auc_siblings_vs_different_

In [7]:
# Figure 6: agreement predicts benefit (matched-volume rescue by kappa band; rescue by relationship)
BANDS = [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={'width_ratios': [1, 1.15]}); BEN = {}
for site in SITES:
    S = DOM[site]; s = PR[(PR.site == site) & (PR['class'].isin(list(S['classes'])))]
    allc = s.groupby(pd.cut(s.kappa, BANDS), observed=False).mv_rescue_share.agg(['median', 'count']).reset_index()
    allc = allc[allc['count'] >= 10]; mid = [iv.mid for iv in allc.kappa]
    axes[0].plot(mid, 100 * allc['median'], ls=LS[site], marker=MARK[site], color=DOMCOL[SDOM[site]], lw=1.8, ms=6, mec='white', label=SNAME[site])
    BEN[site] = {str(k): {'median_rescue': round(float(m), 3), 'pairs': int(c)} for k, m, c in zip(allc.kappa, allc['median'], allc['count'])}
ax = axes[0]; ax.set_xlabel('Referral agreement with the deployed system (kappa, banded)'); ax.set_ylabel('Median % of misses rescued at matched volume')
ax.set_ylim(bottom=0); ax.legend(fontsize=7.5, loc='upper right')
panel(ax, 'a', 'The more a candidate agrees, the less it adds', 'All target conditions; bands with at least 10 pairs')
rels = PR.groupby(['site', 'relationship']).mv_rescue_share.median().unstack(0)
ax = axes[1]; off = dict(zip(SITES, np.linspace(-0.28, 0.28, len(SITES))))
for i, rl in enumerate(RELS):
    ax.axhspan(i - 0.42, i + 0.42, color=RELCOL[rl], alpha=0.08, lw=0)
    for site in SITES:
        raw = [k for k, v in RELMAP.items() if v == rl and k in rels.index and not np.isnan(rels.loc[k, site])]
        for k in raw: ax.scatter(100 * rels.loc[k, site], i + off[site], marker=MARK[site], color=DOMCOL[SDOM[site]], s=46, edgecolor='white', lw=0.7, zorder=3)
ax.set_yticks(range(len(RELS))); ax.set_yticklabels(RELS); ax.set_ylim(len(RELS) - 0.5, -0.5); ax.tick_params(axis='y', length=0)
ax.set_xlim(left=-0.6); ax.set_xlabel('Median % of misses rescued at matched volume')
ax.legend(handles=[Line2D([], [], marker=MARK[s], ls='', color=DOMCOL[SDOM[s]], mec='white', ms=7, label=SNAME[s]) for s in SITES], fontsize=7.5, loc='lower right')
panel(ax, 'b', 'Siblings add least; different data and design add most', 'One point per population; retina systems were all trained on EyePACS except in BRSET')
title(fig, 'Figure 6. Agreement on local unlabelled images predicts how much a candidate can add')
fig.tight_layout(); save(fig, 'Fig6_agreement_benefit')

saved Fig6_agreement_benefit


In [8]:
# Supplementary Figure 9: priced false referrals, the two bounds (from notebook 18b's saved curves; aggregate numbers only)
CUR = pd.read_csv(f'{RESULTS}/bounds_B_tuned_K_v2.csv'); BEK = pd.read_csv(f'{RESULTS}/bounds_B_breakeven_K_v2.csv')
BOUNDS = [('Mathematically conservative: better system of the pair, literature harm', 'better', 'literature harm (central)', BLUE, '-'),
          ('Clinically conservative: deployed system, every missed severe eye blinds', 'deployed', 'every missed severe eye blinds', VERM, '--')]
POPS = [('eyepacs', 'EyePACS (US)'), ('ddr', 'DDR (China)'), ('brset', 'BRSET (Brazil)')]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.6), sharex=True)
for row, (blab, cmp_, harm, col, ls) in enumerate(BOUNDS):
    for j, (pop, pname) in enumerate(POPS):
        ax = axes[row][j]; c = CUR[(CUR.population == pop) & (CUR.comparator == cmp_) & (CUR.harm == harm)].sort_values('K')
        ax.axvspan(800, 3200, color='#BDBDBD', alpha=0.35, lw=0)
        ax.fill_between(c.K, c.ci_lo, c.ci_hi, color=col, alpha=0.18, lw=0); ax.plot(c.K, c.advantage, color=col, ls=ls, lw=2)
        ax.axhline(0, color='#555555', lw=0.8); ax.set_xscale('log')
        b = BEK[(BEK.population == pop) & (BEK.comparator == cmp_) & (BEK.harm == harm)].iloc[0]
        ax.set_title(pname, loc='left', fontsize=9, weight='bold', pad=14)
        lab_ = 'Pair ahead at every K tested, across the 95% interval' if b.ci == 'below 10 to below 10' else f'Break-even K: {b.breakeven_K} (95% CI {b.ci})'
        ax.text(0, 1.015, lab_, transform=ax.transAxes, fontsize=7.8, color='#444444', va='bottom')
        if row == 1: ax.set_xlabel('K: false referrals accepted to prevent one blind eye')
        if j == 0: ax.set_ylabel('Pair minus comparator\n(blind eyes per 10,000 screened)')
    axes[row][0].annotate(f"{'ab'[row]}  {blab}", xy=(0, 1), xycoords='axes fraction', xytext=(-55, 34), textcoords='offset points', fontsize=9.5, weight='bold', ha='left')
fig.legend(handles=[Patch(color='#BDBDBD', alpha=0.6, label='One specialist visit under the only-eye QALY valuation (K about 800 to 3,200)'),
                    Patch(color=GREY, alpha=0.3, label='95% interval (resampling patients or image groups)')], loc='lower left', bbox_to_anchor=(0.01, -0.02), fontsize=8, ncol=2)
title(fig, 'Supplementary Figure 9. Priced false referrals: each strategy tunes its own referral volume (above 0, the pair is ahead)')
fig.tight_layout(rect=(0, 0.04, 1, 0.96)); fig.subplots_adjust(hspace=0.6); save(fig, 'SuppFig9_priced_false_referrals')

saved SuppFig9_priced_false_referrals


In [9]:
# Check: the recomputed numbers equal the ones behind the previous figures
OLD = json.load(open(f'{OUT}/final_figures_data_v2.json'))
ok = abs(OLD['miss_rate_law_spearman'] - round(float(rho), 3)) < 1e-9 and OLD['omitted_points'] == DROPPED and OLD['sibling_detector'] == DET and OLD['benefit_by_kappa_band'] == {k: BEN[k] for k in OLD['benefit_by_kappa_band']}
print('same numbers as final_figures_data_v2.json:', ok)
for k in F3: print(k, 'same minus different data:', ci_txt(F3[k]['mean'], *F3[k]['ci95']))
drive.flush_and_unmount(); drive.mount('/content/drive')
print('Wrote 05_figures/manuscript_final: Fig3 to Fig6 and SuppFig9; previous versions renamed SUPERSEDED_')

same numbers as final_figures_data_v2.json: True
HIBA_mel same minus different data: 0.44 (0.23 to 0.75)
HIBA_mal same minus different data: 0.88 (0.60 to 1.20)
PAD_mal same minus different data: 0.45 (0.37 to 0.54)
Mounted at /content/drive
Wrote 05_figures/manuscript_final: Fig3 to Fig6 and SuppFig9; previous versions renamed SUPERSEDED_
